## Day 9 — Recommendation System

**What we are building:** For every user, find the top 5 products they are most likely to buy next.

**Algorithm:** ALS (Alternating Least Squares) — collaborative filtering

**Input:** `ecommerce.silver.events_clean`  
**Output:** `ecommerce.gold.user_recommendations`

---
**Why ALS needs integers:**  
ALS builds a matrix: rows = users, columns = products.  
Matrix positions must be integers (0, 1, 2...) — UUID strings won't work.  
So we convert: `'abc-123...'` → `0`, `'def-456...'` → `1`, etc.

**Run Cell 1 first after any cluster reset.**

In [0]:
# Everything we need for Day 9
from pyspark.ml.recommendation import ALS
from pyspark.ml.feature import StringIndexer
from pyspark.ml.evaluation import RegressionEvaluator
import pyspark.sql.functions as F

CATALOG      = 'ecommerce'
OUTPUT_TABLE = 'ecommerce.gold.user_recommendations'

print('Ready')


Ready


### Step 1 — Load the raw events
Each row = one user doing something to one product (view, cart, or purchase).

In [0]:
# Load all cleaned events from Silver layer
events = spark.table('ecommerce.silver.events')

print(f'Total events: {events.count():,}')
print()

# See what event types exist
events.groupBy('event_type').count().show()

# Preview a few rows
display(events.select('user_id', 'product_id', 'event_type').limit(5))


Total events: 109,806,507

+----------+---------+
|event_type|    count|
+----------+---------+
|  purchase|  1659703|
|      cart|  3828450|
|      view|104318354|
+----------+---------+



user_id,product_id,event_type
523084115,3601278,view
532110825,1005099,view
519486709,5100835,view
519064170,28719204,view
562778657,48200034,view


###  Give each event a score

We can't feed raw events into ALS. We need ONE number per user-product pair that says how strongly that user likes that product.

- **purchase = 3** → user actually bought it (strongest signal)
- **cart = 2** → user intended to buy
- **view = 1** → user was mildly interested

If a user viewed the same product 5 times and bought it once → score = `5×1 + 1×3 = 8`

In [0]:
# Step 1: Give each event a weight
events_scored = events.withColumn('weight',
    F.when(F.col('event_type') == 'purchase', 3)
     .when(F.col('event_type') == 'cart',     2)
     .when(F.col('event_type') == 'view',      1)
     .otherwise(0)
)

# Step 2: Add up all weights per user-product pair
# One user can interact with the same product many times
# We want ONE total score per pair, not one row per event
interactions = (
    events_scored
    .groupBy('user_id', 'product_id')
    .agg(F.sum('weight').alias('score'))
)

print(f'Unique user-product pairs: {interactions.count():,}')
print('Highest scores (most engaged users):')
display(interactions.orderBy(F.desc('score')).limit(10))


Unique user-product pairs: 56,371,590
Highest scores (most engaged users):


user_id,product_id,score
564068124,1004833,1701
523974502,5100563,1511
564068124,1004767,1207
549030056,1004856,1018
543128872,4804056,978
535658024,1002544,932
541510103,1004833,905
518514099,1005116,870
512386086,1801555,864
552639168,4100346,835


### Step 3 — Convert string IDs to integers

ALS needs integer IDs. `StringIndexer` scans all user IDs, sorts them by frequency, and assigns each one a number:
most common user → 0, next → 1, and so on.

We save the indexer models (`user_idx` and `product_idx`) because we need them later to convert numbers **back** to real IDs.

In [0]:
# ALS needs integer IDs — but StringIndexer crashes on serverless
# with 5.7M users (model too large for Spark Connect)
#
# Fix: use dense_rank() window function instead
# dense_rank() assigns 1, 2, 3... to each unique value — no model needed
# Works entirely as a DataFrame operation — no size limit

from pyspark.sql import Window

# Rank users: each unique user_id gets a unique integer
user_window    = Window.orderBy('user_id')
product_window = Window.orderBy('product_id')

interactions = interactions \
    .withColumn('user_int', F.dense_rank().over(user_window).cast('integer') - 1) \
    .withColumn('product_int', F.dense_rank().over(product_window).cast('integer') - 1)

# Save the lookup tables BEFORE we lose the string IDs
# We need these in Cell 9 to convert integer recommendations back to real IDs
user_lookup    = interactions.select('user_id',    'user_int').distinct()
product_lookup = interactions.select('product_id', 'product_int').distinct()

print(f'Unique users:    {interactions.select("user_int").distinct().count():,}')
print(f'Unique products: {interactions.select("product_int").distinct().count():,}')
print()
display(interactions.select('user_id', 'product_id', 'score', 'user_int', 'product_int').limit(5))


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1134: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Unique users:    5,316,649
Unique products: 206,876



user_id,product_id,score,user_int,product_int
525766088,1000365,1,1314271,0
442462854,1000978,8,7385,1
424872790,1000978,1,4765,1
460216566,1000978,1,11658,1
460263198,1000978,2,11684,1


### Step 4 — Split into train and test

Same pattern as Day 5. Train on 80%, test on 20%. The test set lets us measure RMSE — how far off our predictions are.

In [0]:
train, test = interactions.randomSplit([0.8, 0.2], seed=42)

print(f'Train: {train.count():,} pairs')
print(f'Test:  {test.count():,} pairs')


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1134: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Train: 45,100,640 pairs
Test:  11,270,950 pairs


###  Train the ALS Model

ALS learns a 10-number fingerprint for every user and every product. Multiply a user's fingerprint by a product's fingerprint → predicted preference score.

Key setting: `implicitPrefs=True` — because we have purchase **events**, not star ratings. ALS treats the score as a confidence level, not an explicit preference.

In [0]:
als = ALS(
    userCol           = 'user_int',    # integer user column
    itemCol           = 'product_int', # integer product column
    ratingCol         = 'score',       # our interaction score
    rank              = 10,            # fingerprint size (10 numbers per user/product)
    maxIter           = 10,            # how many training rounds
    regParam          = 0.1,           # prevents overfitting
    implicitPrefs     = True,          # we have events, not star ratings
    coldStartStrategy = 'drop',        # skip users not in training (avoids NaN errors)
    seed              = 42
)

print('Training ALS...')
als_model = als.fit(train)
print('Done!')
print(f'Learned fingerprints for {als_model.userFactors.count():,} users and {als_model.itemFactors.count():,} products')


Training ALS...


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1134: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Done!
Learned fingerprints for 4,952,215 users and 203,762 products


In [0]:
als_model.recommendForAllUsers(5).show(5)

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-5129662071489018>, line 1
----> 1 als_model.recommendForAllUsers(5).show(5)

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/dataframe.py:1156, in DataFrame.show(self, n, truncate, vertical)
   1155 def show(self, n: int = 20, truncate: Union[bool, int] = True, vertical: bool = False) -> None:
-> 1156     print(self._show_string(n, truncate, vertical))

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/dataframe.py:909, in DataFrame._show_string(self, n, truncate, vertical)
    892     except ValueError:
    893         raise PySparkTypeError(
    894             errorClass="NOT_BOOL",
    895             messageParameters={
   (...)
    898             },
    899         )
    901 table, _ = DataFrame(
    902     plan.ShowString(
    903         child=self._plan,
    904   

### Step 5 — Check quality (RMSE)

RMSE = average gap between predicted and actual score on the test set. Lower is better. For event data, 0.5–2.0 is a typical range.

In [0]:
# Run model on test set to get predictions
test_preds = als_model.transform(test)

# Measure RMSE
rmse = RegressionEvaluator(
    metricName='rmse', labelCol='score', predictionCol='prediction'
).evaluate(test_preds)

print(f'RMSE: {rmse:.4f}')
print('Lower = better. Typical range for implicit data: 0.5 to 2.0')
print()

# Show predicted vs actual for a few pairs
display(test_preds.select('user_id', 'product_id', 'score', 'prediction').limit(10))


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1134: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


RMSE: 3.9011
Lower = better. Typical range for implicit data: 0.5 to 2.0



user_id,product_id,score,prediction
66082846,31501195,2,2.1671012E-6
85786403,25200350,2,3.423996E-8
107837897,4700557,7,0.012377519
107837897,4700605,3,0.0020856098
124298297,49100001,1,1.6438567E-4
124298297,49100002,5,1.1493524E-4
124298297,49100004,2,1.486634E-4
124298297,49100015,1,1.0803655E-4
178841989,3100662,1,7.2952756E-4
178841989,11200421,1,1.7667913E-5


### Task 3 — Generate Top-5 Recommendations

`recommendForAllUsers(5)` runs through every user and picks the 5 products they are most likely to buy. The raw output looks like this:

```
user_int=0  recommendations=[{product_int:42, rating:0.9}, {product_int:7, rating:0.8}, ...]
user_int=1  recommendations=[{product_int:15, rating:0.95}, ...]
```

We then flatten this and convert the integer IDs back to real product IDs.

In [0]:
# Generate top 5 products for every user
# Returns one row per user with an array of 5 recommendations
print('Generating recommendations...')
top5_raw = als_model.recommendForAllUsers(5)
print(f'Done! Recommendations for {top5_raw.count():,} users')
print()
print('Raw format — array of {product_int, rating} for each user:')
display(top5_raw.limit(3))


Generating recommendations...


---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-5129662071488983>, line 5
      3 print('Generating recommendations...')
      4 top5_raw = als_model.recommendForAllUsers(5)
----> 5 print(f'Done! Recommendations for {top5_raw.count():,} users')
      6 print()
      7 print('Raw format — array of {product_int, rating} for each user:')

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/dataframe.py:318, in DataFrame.count(self)
    315 def count(self) -> int:
    316     table, _ = self.agg(
    317         F._invoke_function("count", F.lit(1))
--> 318     )._to_table()  # type: ignore[operator]
    319     return table[0][0].as_py()

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/dataframe.py:1923, in DataFrame._to_table(self)
   1921 def _to_table(self) -> Tuple["pa.Table", Optional[StructType]]:
   1922     query = self

In [0]:
# Cell 8: rename factor columns so they don't clash after cross join
user_factors = als_model.userFactors.withColumnRenamed('features', 'user_vec')
prod_factors = als_model.itemFactors.withColumnRenamed('features', 'prod_vec')

RANK = 10 
# Cell 9: cross join + dot product as a column expression
dot_expr = sum(F.col('user_vec')[k] * F.col('prod_vec')[k] for k in range(RANK))

scored = user_factors.crossJoin(prod_factors).withColumn('rec_score', dot_expr)

# Top 5 per user — partitionBy(user_int) avoids the global sort warning
w = Window.partitionBy('user_int').orderBy(F.desc('rec_score'))
top5_flat = scored.withColumn('rank', F.row_number().over(w)).filter('rank <= 5')

In [0]:
# recommendForAllUsers() — blocked by Unity Catalog
# sparkContext / RDD broadcast — blocked on serverless
#
# Fix: pure DataFrame dot product
#
# ALS learned a 10-number vector for every user and every product.
# Score = sum of (user_vector[i] * product_vector[i]) for i in 0..9
# This is a dot product — we compute it using DataFrame column math only.
#
# Steps:
#   1. Rename factor columns so user and product don't clash
#   2. Cross join every user with every product
#   3. Compute dot product as a SQL expression
#   4. Keep top 5 per user using row_number() partitioned by user

from pyspark.sql.types import DoubleType
import pyspark.sql.functions as F
from pyspark.sql import Window

RANK = als_model.rank   # number of factors (10)

# Step 1: Pull factor tables and rename columns to avoid clashes
# userFactors  schema: id (int), features (array<float>)
# itemFactors  schema: id (int), features (array<float>)
user_factors = als_model.userFactors.withColumnRenamed('id', 'user_int') \
                                    .withColumnRenamed('features', 'user_vec')

prod_factors = als_model.itemFactors.withColumnRenamed('id', 'product_int') \
                                    .withColumnRenamed('features', 'prod_vec')

print(f'Users:    {user_factors.count():,}')
print(f'Products: {prod_factors.count():,}')
print('Ready for cross join + dot product')

Users:    4,952,215
Products: 203,762
Ready for cross join + dot product


### Step 6 — Flatten and decode to real IDs

Two things to do:
1. **Explode** — turn the array of 5 into 5 separate rows
2. **Decode** — convert integer IDs back to the original string user/product IDs

In [0]:
RANK = 10  # number of ALS factors — must match rank= in Cell 6

# Step 2: Cross join — every user paired with every product
# Then compute dot product using array element multiplication + sum
#
# dot_product = user_vec[0]*prod_vec[0] + user_vec[1]*prod_vec[1] + ... + user_vec[9]*prod_vec[9]
# We build this as a column expression using a Python loop over the rank

# Build dot product expression: sum of element-wise products
dot_expr = sum(
    F.col('user_vec')[k] * F.col('prod_vec')[k]
    for k in range(RANK)
).cast(DoubleType())

# Cross join + score (products are small enough to broadcast automatically)
scored = (
    user_factors
    .crossJoin(prod_factors)
    .withColumn('rec_score', dot_expr)
    .select('user_int', 'product_int', 'rec_score')
)

# Step 3: Keep top 5 per user using row_number partitioned by user
# partitionBy(user_int) means each user's rows stay together — no global sort
w = Window.partitionBy('user_int').orderBy(F.desc('rec_score'))

top5_flat = (
    scored
    .withColumn('rank', F.row_number().over(w))
    .filter(F.col('rank') <= 5)
    .drop('rank')
)

# Step 4: Decode integer IDs back to real string IDs
top5 = (
    top5_flat
    .join(user_lookup,    'user_int')
    .join(product_lookup, 'product_int')
    .select('user_id', 'product_id', 'rec_score')
)

print(f'Total recommendation rows: {top5.count():,}  (should be users x 5)')
display(top5.limit(10))


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1134: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


com.databricks.backend.common.rpc.CommandCancelledException
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$5(SequenceExecutionState.scala:139)
	at scala.Option.getOrElse(Option.scala:201)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3(SequenceExecutionState.scala:139)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3$adapted(SequenceExecutionState.scala:136)
	at scala.collection.immutable.Range.foreach(Range.scala:192)
	at com.databricks.spark.chauffeur.SequenceExecutionState.cancel(SequenceExecutionState.scala:136)
	at com.databricks.spark.chauffeur.ExecContextState.cancelRunningSequence(ExecContextState.scala:724)
	at com.databricks.spark.chauffeur.ExecContextState.$anonfun$cancel$1(ExecContextState.scala:442)
	at scala.Option.getOrElse(Option.scala:201)
	at com.databricks.spark.chauffeur.ExecContextState.cancel(ExecContextState.scala:442)
	at com.databricks.spark.chauffeur.ExecutionContextManagerV1.can

### Step 7 — Save to Gold Delta Table

Same `writeTo().createOrReplace()` pattern as Day 8 — serverless compatible.

In [0]:
# Add timestamp so we know when recommendations were generated
top5_final = top5.withColumn('generated_at', F.current_timestamp())

# Save — writeTo() is serverless compatible (saveAsTable is not)
(
    top5_final
    .writeTo(OUTPUT_TABLE)
    .using('delta')
    .createOrReplace()
)

print(f'Saved {spark.table(OUTPUT_TABLE).count():,} rows to {OUTPUT_TABLE}')
display(spark.table(OUTPUT_TABLE).limit(10))


### Step 8 — Check the results

See the top 5 recommended products for a few real users.

In [0]:
df_recs = spark.table(OUTPUT_TABLE)

# Show top 5 recommendations for 3 sample users
sample_users = [r.user_id for r in df_recs.select('user_id').distinct().limit(3).collect()]

for uid in sample_users:
    print(f'User: {uid}')
    df_recs.filter(F.col('user_id') == uid) \
           .orderBy(F.desc('rec_score')) \
           .select('product_id', 'rec_score') \
           .show(5, truncate=False)

# Quick summary
print('=== SUMMARY ===')
df_recs.agg(
    F.countDistinct('user_id').alias('users_with_recommendations'),
    F.countDistinct('product_id').alias('unique_products_recommended'),
    F.round(F.avg('rec_score'), 4).alias('avg_recommendation_score')
).show()

print('Gold layer complete:')
print('  ecommerce.gold.user_features         -> who users are')
print('  ecommerce.gold.user_predictions      -> will they buy?')
print('  ecommerce.gold.user_recommendations  -> what to show them')
print()
print('9-Day Databricks AI Challenge COMPLETE!')


## Done!

| Cell | What it does |
|------|--------------|
| 1 | Imports |
| 2 | Load raw events from Silver |
| 3 | Build interaction scores: purchase=3, cart=2, view=1 |
| 4 | Convert string IDs → integers (ALS requires this) |
| 5 | 80/20 train/test split |
| 6 | Train ALS model |
| 7 | Evaluate RMSE |
| 8 | Generate Top-5 recommendations per user |
| 9 | Flatten array + decode integers back to real IDs |
| 10 | Save to `ecommerce.gold.user_recommendations` |
| 11 | Inspect results |
